# Tutorial: Registro de Imágenes (Image Registration)

**Curso:** Visión por Computador 3009228 - Universidad Nacional de Colombia  
**Docente:** Juan David Ospina Arango

---

## Introducción

El **registro de imágenes** es el proceso de alinear dos o más imágenes de la misma escena. Este tutorial cubre:

1. 🎯 Conceptos fundamentales del registro
2. 🔧 Tipos de transformaciones geométricas
3. 📊 Métodos basados en intensidad
4. 🔍 Métodos basados en características
5. 📈 Métricas de similitud
6. 💻 Implementaciones prácticas con Python

In [1]:
# Importar bibliotecas necesarias
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
import warnings
warnings.filterwarnings('ignore')

# Configuración para mejores visualizaciones
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['font.size'] = 10

print("✅ Bibliotecas importadas correctamente")
print(f"OpenCV version: {cv2.__version__}")
print(f"NumPy version: {np.__version__}")

/Users/jdospina/anaconda3/envs/ml_curso/lib/python3.11/site-packages/scipy/__init__.py:132: UserWarning: A NumPy version >=1.21.6 and <1.28.0 is required for this version of SciPy (detected version 2.2.6)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/jdospina/anaconda3/envs/ml_curso/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_ins

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

## 1. Fundamentos del Registro de Imágenes

### ¿Qué es el Registro de Imágenes?

El registro busca encontrar una **transformación geométrica** $T$ que mapee los píxeles de una imagen móvil $I_m$ a las coordenadas correspondientes en una imagen fija $I_f$:

$$I_f(x, y) \approx I_m(T(x, y))$$

### Componentes Principales:

1. **Imagen fija** ($I_f$): la imagen de referencia
2. **Imagen móvil** ($I_m$): la imagen a transformar
3. **Transformación** ($T$): función de mapeo de coordenadas
4. **Métrica de similitud** ($S$): mide la calidad del alineamiento
5. **Optimizador**: busca la mejor transformación

### Aplicaciones:

- 🏥 **Imágenes médicas**: alinear scans de diferentes modalidades
- 🛰️ **Teledetección**: monitorear cambios temporales
- 📷 **Panorámicas**: unir múltiples imágenes
- 👁️ **Visión estéreo**: correspondencias entre vistas

In [ ]:
# Función para crear imágenes sintéticas
def crear_imagen_sintetica(size=256, tipo='patron'):
    """
    Crea imágenes sintéticas para demostración.
    
    Args:
        size: tamaño de la imagen (size x size)
        tipo: 'patron', 'cuadros', 'circulo', 'texto'
    
    Returns:
        imagen en escala de grises
    """
    imagen = np.zeros((size, size), dtype=np.uint8)
    
    if tipo == 'patron':
        # Patrón de tablero de ajedrez
        square_size = size // 8
        for i in range(8):
            for j in range(8):
                if (i + j) % 2 == 0:
                    imagen[i*square_size:(i+1)*square_size, 
                          j*square_size:(j+1)*square_size] = 255
        
        cv2.circle(imagen, (size//2, size//2), size//6, 128, -1)
        cv2.rectangle(imagen, (size//4, size//4), (3*size//4, size//3), 200, 3)
        
    elif tipo == 'cuadros':
        cv2.rectangle(imagen, (20, 20), (100, 100), 255, -1)
        cv2.rectangle(imagen, (size-120, 40), (size-40, 120), 180, -1)
        cv2.rectangle(imagen, (size//2-30, size-80), (size//2+30, size-20), 220, -1)
        
    elif tipo == 'circulo':
        cv2.circle(imagen, (size//2, size//2), size//3, 255, -1)
        cv2.circle(imagen, (size//2, size//2), size//4, 0, -1)
        cv2.circle(imagen, (size//2, size//2), size//6, 200, -1)
        
    elif tipo == 'texto':
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(imagen, 'CV', (size//4, size//2), font, 3, 255, 5)
        cv2.circle(imagen, (3*size//4, size//4), 30, 255, -1)
        cv2.circle(imagen, (3*size//4, 3*size//4), 40, 200, -1)
    
    return imagen

# Demostración
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tipos = ['patron', 'cuadros', 'circulo', 'texto']

for ax, tipo in zip(axes, tipos):
    img = crear_imagen_sintetica(256, tipo)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Tipo: {tipo}', fontsize=12)
    ax.axis('off')

plt.suptitle('Imágenes Sintéticas para Demostración', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Transformaciones Geométricas

Las transformaciones geométricas forman una jerarquía basada en el número de grados de libertad (DOF):

| Transformación | DOF | Parámetros | Propiedades Preservadas |
|---------------|-----|------------|------------------------|
| **Traslación** | 2 | $t_x, t_y$ | Distancias, ángulos, áreas |
| **Rotación** | 1 | $\theta$ | Distancias, ángulos, áreas |
| **Rígida** (Euclidiana) | 3 | $t_x, t_y, \theta$ | Distancias, ángulos, áreas |
| **Similitud** | 4 | $t_x, t_y, \theta, s$ | Ángulos, ratios de áreas |
| **Afín** | 6 | 6 parámetros | Paralelismo, ratios de distancias |
| **Proyectiva** | 8 | 8 parámetros (homografía) | Líneas rectas |

### Representación Matricial:

**Transformación Afín** (2D):
$$
\begin{bmatrix} x' \\ y' \end{bmatrix} = 
\begin{bmatrix} a & b \\ c & d \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix} +
\begin{bmatrix} t_x \\ t_y \end{bmatrix}
$$

**Transformación Proyectiva** (homografía):
$$
\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} \sim 
\begin{bmatrix} 
h_{11} & h_{12} & h_{13} \\ 
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix}
\begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

In [ ]:
# Función para aplicar transformaciones
def aplicar_transformacion(imagen, tipo, params):
    """
    Aplica una transformación geométrica a una imagen.
    """
    h, w = imagen.shape[:2]
    center = (w / 2, h / 2)
    
    if tipo == 'traslacion':
        tx = params.get('tx', 0)
        ty = params.get('ty', 0)
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        imagen_trans = cv2.warpAffine(imagen, M, (w, h))
        
    elif tipo == 'rotacion':
        angulo = params.get('angulo', 0)
        M = cv2.getRotationMatrix2D(center, angulo, 1.0)
        imagen_trans = cv2.warpAffine(imagen, M, (w, h))
        
    elif tipo == 'rigida':
        angulo = params.get('angulo', 0)
        tx = params.get('tx', 0)
        ty = params.get('ty', 0)
        M = cv2.getRotationMatrix2D(center, angulo, 1.0)
        M[0, 2] += tx
        M[1, 2] += ty
        imagen_trans = cv2.warpAffine(imagen, M, (w, h))
        
    elif tipo == 'afin':
        angulo = params.get('angulo', 0)
        escala = params.get('escala', 1.0)
        tx = params.get('tx', 0)
        ty = params.get('ty', 0)
        shear = params.get('shear', 0)
        
        M = cv2.getRotationMatrix2D(center, angulo, escala)
        M[0, 2] += tx
        M[1, 2] += ty
        M[0, 1] += shear
        imagen_trans = cv2.warpAffine(imagen, M, (w, h))
    
    else:
        raise ValueError(f"Tipo '{tipo}' no reconocido")
    
    return imagen_trans, M

# Demostración de transformaciones
img_base = crear_imagen_sintetica(256, 'texto')

transformaciones = {
    'Original': img_base,
    'Traslación\n(tx=30, ty=20)': aplicar_transformacion(img_base, 'traslacion', {'tx': 30, 'ty': 20})[0],
    'Rotación\n(30°)': aplicar_transformacion(img_base, 'rotacion', {'angulo': 30})[0],
    'Rígida\n(θ=20°, tx=15, ty=-10)': aplicar_transformacion(img_base, 'rigida', 
                                      {'angulo': 20, 'tx': 15, 'ty': -10})[0],
    'Afín\n(θ=15°, s=0.8, shear=0.2)': aplicar_transformacion(img_base, 'afin',
                                       {'angulo': 15, 'escala': 0.8, 'tx': 10, 'ty': 5, 'shear': 0.2})[0]
}

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (titulo, img) in zip(axes, transformaciones.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(titulo, fontsize=11)
    ax.axis('off')

plt.suptitle('Tipos de Transformaciones Geométricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Métricas de Similitud

Las métricas de similitud cuantifican qué tan bien están alineadas dos imágenes:

### 1. Error Cuadrático Medio (MSE)
$$\text{MSE}(I_f, I_m) = \frac{1}{N}\sum_{i=1}^{N}(I_f(i) - I_m(i))^2$$
- **Objetivo**: Minimizar
- **Ventajas**: Simple, rápido
- **Desventajas**: Sensible a diferencias de brillo y outliers

### 2. Coeficiente de Correlación Cruzada Normalizada (NCC)
$$\text{NCC}(I_f, I_m) = \frac{\sum(I_f - \bar{I_f})(I_m - \bar{I_m})}{\sqrt{\sum(I_f - \bar{I_f})^2 \sum(I_m - \bar{I_m})^2}}$$
- **Objetivo**: Maximizar
- **Ventajas**: Invariante a cambios lineales de brillo
- **Desventajas**: Mayor costo computacional

### 3. Información Mutua (MI)
$$\text{MI}(I_f, I_m) = \sum_{i,j} p(i,j)\log\frac{p(i,j)}{p(i)p(j)}$$
- **Objetivo**: Maximizar
- **Ventajas**: Funciona con imágenes multi-modales
- **Desventajas**: Computacionalmente costoso

In [ ]:
# Implementación de métricas de similitud
def calcular_similitud(img1, img2, metrica='mse'):
    """
    Calcula una métrica de similitud entre dos imágenes.
    """
    if metrica == 'mse':
        mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
        return mse
    
    elif metrica == 'ncc':
        img1_norm = (img1 - np.mean(img1)) / (np.std(img1) + 1e-8)
        img2_norm = (img2 - np.mean(img2)) / (np.std(img2) + 1e-8)
        ncc = np.mean(img1_norm * img2_norm)
        return ncc
    
    elif metrica == 'mi':
        hist_2d, _, _ = np.histogram2d(img1.ravel(), img2.ravel(), bins=20)
        pxy = hist_2d / float(np.sum(hist_2d))
        px = np.sum(pxy, axis=1)
        py = np.sum(pxy, axis=0)
        px_py = px[:, None] * py[None, :]
        nzs = pxy > 0
        mi = np.sum(pxy[nzs] * np.log(pxy[nzs] / px_py[nzs]))
        return mi
    
    else:
        raise ValueError(f"Métrica '{metrica}' no reconocida")

# Demostración de métricas
img1 = crear_imagen_sintetica(256, 'patron')
img2_alineada = img1.copy()
img2_desalineada, _ = aplicar_transformacion(img1, 'traslacion', {'tx': 10, 'ty': 5})
img2_muy_desalineada, _ = aplicar_transformacion(img1, 'traslacion', {'tx': 30, 'ty': 20})

imagenes_test = {
    'Perfectamente alineada': img2_alineada,
    'Ligeramente desalineada': img2_desalineada,
    'Muy desalineada': img2_muy_desalineada
}

print("\n" + "="*60)
print("COMPARACIÓN DE MÉTRICAS DE SIMILITUD")
print("="*60)

for nombre, img2 in imagenes_test.items():
    mse = calcular_similitud(img1, img2, 'mse')
    ncc = calcular_similitud(img1, img2, 'ncc')
    mi = calcular_similitud(img1, img2, 'mi')
    
    print(f"\n{nombre}:")
    print(f"  MSE: {mse:.2f} (menor es mejor)")
    print(f"  NCC: {ncc:.4f} (mayor es mejor)")
    print(f"  MI:  {mi:.4f} (mayor es mejor)")

# Visualización
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img1, cmap='gray')
axes[0].set_title('Imagen de Referencia', fontsize=12)
axes[0].axis('off')

for ax, (nombre, img) in zip(axes[1:], imagenes_test.items()):
    ax.imshow(img, cmap='gray')
    ax.set_title(nombre, fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Registro Basado en Intensidad: Búsqueda Exhaustiva

El método más simple es la **búsqueda exhaustiva**:
1. Definir un espacio de búsqueda de parámetros
2. Para cada combinación de parámetros:
   - Aplicar la transformación
   - Calcular la similitud
3. Seleccionar la transformación con mejor similitud

**Ventajas:**
- ✅ Simple de implementar
- ✅ Garantiza encontrar el óptimo en el espacio de búsqueda

**Desventajas:**
- ❌ Computacionalmente costoso
- ❌ No escalable a muchos parámetros

In [ ]:
def registro_busqueda_exhaustiva(img_fija, img_movil, 
                                 rango_tx=(-20, 20), 
                                 rango_ty=(-20, 20),
                                 paso=1,
                                 metrica='mse'):
    """
    Registro por búsqueda exhaustiva de traslación.
    """
    mejor_similitud = float('inf') if metrica == 'mse' else float('-inf')
    mejor_tx, mejor_ty = 0, 0
    historia = []
    
    total_iteraciones = ((rango_tx[1] - rango_tx[0]) // paso + 1) * \
                       ((rango_ty[1] - rango_ty[0]) // paso + 1)
    
    print(f"Explorando {total_iteraciones} combinaciones...")
    
    for tx in range(rango_tx[0], rango_tx[1] + 1, paso):
        for ty in range(rango_ty[0], rango_ty[1] + 1, paso):
            img_trans, _ = aplicar_transformacion(img_movil, 'traslacion', {'tx': tx, 'ty': ty})
            sim = calcular_similitud(img_fija, img_trans, metrica)
            historia.append((tx, ty, sim))
            
            if metrica == 'mse':
                if sim < mejor_similitud:
                    mejor_similitud = sim
                    mejor_tx, mejor_ty = tx, ty
            else:
                if sim > mejor_similitud:
                    mejor_similitud = sim
                    mejor_tx, mejor_ty = tx, ty
    
    M_optima = np.float32([[1, 0, mejor_tx], [0, 1, mejor_ty]])
    return M_optima, mejor_similitud, historia

# EJEMPLO: Registro de traslación
print("\n" + "="*60)
print("EJEMPLO: REGISTRO DE TRASLACIÓN POR BÚSQUEDA EXHAUSTIVA")
print("="*60 + "\n")

# Crear imagen y aplicar traslación conocida
img_fija = crear_imagen_sintetica(256, 'patron')
traslacion_real = {'tx': 15, 'ty': -10}
img_movil, _ = aplicar_transformacion(img_fija, 'traslacion', traslacion_real)

print(f"Traslación aplicada: tx={traslacion_real['tx']}, ty={traslacion_real['ty']}")
print("Buscando la traslación óptima...\n")

# Realizar registro
M_optima, similitud, historia = registro_busqueda_exhaustiva(
    img_fija, img_movil,
    rango_tx=(-30, 30),
    rango_ty=(-30, 30),
    paso=1,
    metrica='mse'
)

tx_encontrado = M_optima[0, 2]
ty_encontrado = M_optima[1, 2]

print(f"\n✅ Traslación encontrada: tx={tx_encontrado:.1f}, ty={ty_encontrado:.1f}")
print(f"Error: Δx={abs(tx_encontrado - traslacion_real['tx']):.1f}, "
      f"Δy={abs(ty_encontrado - traslacion_real['ty']):.1f} píxeles")

# Aplicar transformación óptima
img_registrada = cv2.warpAffine(img_movil, M_optima, (img_fija.shape[1], img_fija.shape[0]))

# Visualizar resultados
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(img_fija, cmap='gray')
axes[0, 0].set_title('Imagen Fija', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_movil, cmap='gray')
axes[0, 1].set_title('Imagen Móvil (Original)', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(img_registrada, cmap='gray')
axes[0, 2].set_title('Imagen Móvil (Registrada)', fontsize=12)
axes[0, 2].axis('off')

# Superposiciones
superposicion_antes = np.zeros((*img_fija.shape, 3), dtype=np.uint8)
superposicion_antes[:, :, 0] = img_fija
superposicion_antes[:, :, 1] = img_movil
axes[1, 0].imshow(superposicion_antes)
axes[1, 0].set_title('Antes: Rojo=fija, Verde=móvil', fontsize=12)
axes[1, 0].axis('off')

superposicion_despues = np.zeros((*img_fija.shape, 3), dtype=np.uint8)
superposicion_despues[:, :, 0] = img_fija
superposicion_despues[:, :, 1] = img_registrada
axes[1, 1].imshow(superposicion_despues)
axes[1, 1].set_title('Después: Rojo=fija, Verde=registrada', fontsize=12)
axes[1, 1].axis('off')

diferencia = cv2.absdiff(img_fija, img_registrada)
axes[1, 2].imshow(diferencia, cmap='hot')
axes[1, 2].set_title('Diferencia Absoluta', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Registro por Búsqueda Exhaustiva', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Registro Basado en Características

Un enfoque más eficiente es usar **características locales**:

### Pipeline:
1. **Detectar** puntos de interés (keypoints) en ambas imágenes
2. **Describir** cada punto con un descriptor invariante
3. **Emparejar** descriptores entre imágenes
4. **Estimar** la transformación usando RANSAC

### Detectores Populares:
- **SIFT** (Scale-Invariant Feature Transform): Invariante a escala y rotación
- **ORB** (Oriented FAST and Rotated BRIEF): Rápido, libre de patentes
- **AKAZE**: Buen balance velocidad/precisión

### RANSAC (RANdom SAmple Consensus):
Algoritmo robusto para estimar parámetros en presencia de outliers:
1. Seleccionar aleatoriamente un subconjunto mínimo de datos
2. Calcular el modelo con este subconjunto
3. Contar cuántos puntos son consistentes (inliers)
4. Repetir N veces y quedarse con el mejor modelo

In [ ]:
def registro_con_caracteristicas(img_fija, img_movil, metodo='orb', max_features=500):
    """
    Registro usando detección y emparejamiento de características.
    """
    # Crear detector
    if metodo == 'orb':
        detector = cv2.ORB_create(max_features)
    elif metodo == 'sift':
        detector = cv2.SIFT_create(max_features)
    elif metodo == 'akaze':
        detector = cv2.AKAZE_create()
    else:
        raise ValueError(f"Método '{metodo}' no reconocido")
    
    # Detectar y computar
    kp1, des1 = detector.detectAndCompute(img_fija, None)
    kp2, des2 = detector.detectAndCompute(img_movil, None)
    
    if des1 is None or des2 is None:
        print("⚠️ No se detectaron suficientes características")
        return None, None, None
    
    # Emparejar
    if metodo in ['orb', 'akaze']:
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    else:
        matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    
    matches = matcher.knnMatch(des1, des2, k=2)
    
    # Ratio test de Lowe
    buenos_matches = []
    for m_n in matches:
        if len(m_n) == 2:
            m, n = m_n
            if m.distance < 0.75 * n.distance:
                buenos_matches.append(m)
    
    print(f"✓ Características: {len(kp1)} en img1, {len(kp2)} en img2")
    print(f"✓ Matches buenos: {len(buenos_matches)}")
    
    if len(buenos_matches) < 4:
        print("⚠️ Insuficientes matches para homografía")
        return None, None, None
    
    # Extraer puntos
    pts_fija = np.float32([kp1[m.queryIdx].pt for m in buenos_matches])
    pts_movil = np.float32([kp2[m.trainIdx].pt for m in buenos_matches])
    
    # RANSAC
    H, mask = cv2.findHomography(pts_movil, pts_fija, cv2.RANSAC, 5.0)
    inliers = mask.ravel().tolist()
    print(f"✓ Inliers (RANSAC): {sum(inliers)}/{len(inliers)}")
    
    # Visualizar matches
    img_matches = cv2.drawMatches(
        img_fija, kp1, img_movil, kp2, buenos_matches, None,
        matchesMask=inliers,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    
    return H, (pts_fija, pts_movil, inliers), img_matches

# EJEMPLO: Registro de rotación
print("\n" + "="*60)
print("EJEMPLO: REGISTRO DE ROTACIÓN CON CARACTERÍSTICAS")
print("="*60 + "\n")

img_fija = crear_imagen_sintetica(256, 'cuadros')
angulo_real = 25
img_movil, _ = aplicar_transformacion(img_fija, 'rotacion', {'angulo': angulo_real})

print(f"Rotación aplicada: {angulo_real}°")
print("Detectando características...\n")

H, puntos, img_matches = registro_con_caracteristicas(img_fija, img_movil, metodo='orb')

if H is not None:
    h, w = img_fija.shape
    img_registrada = cv2.warpPerspective(img_movil, H, (w, h))
    
    # Visualizar
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    axes[0, 0].imshow(img_fija, cmap='gray')
    axes[0, 0].set_title('Imagen Fija', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(img_movil, cmap='gray')
    axes[0, 1].set_title(f'Imagen Móvil (rotada {angulo_real}°)', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[1, 0].imshow(img_matches)
    axes[1, 0].set_title('Matches de Características (verde=inliers)', fontsize=12)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(img_registrada, cmap='gray')
    axes[1, 1].set_title('Imagen Registrada', fontsize=12)
    axes[1, 1].axis('off')
    
    plt.suptitle('Registro Basado en Características', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ Registro completado exitosamente")
else:
    print("❌ No se pudo completar el registro")

## 6. Ejercicios Propuestos

### Ejercicio 1: Comparación de Métodos ⭐
Compara el registro por búsqueda exhaustiva vs. características para una imagen con ruido.

### Ejercicio 2: Registro Multi-escala ⭐⭐
Implementa un enfoque piramidal para mejorar la velocidad y robustez del registro.

### Ejercicio 3: Panorámica Simple ⭐⭐⭐
Crea una panorámica uniendo 3 imágenes con superposición.

### Ejercicio 4: Evaluación de Métricas ⭐⭐
Evalúa diferentes métricas de similitud bajo diferentes condiciones (ruido, cambios de iluminación).

### Ejercicio 5: Aplicación Médica ⭐⭐⭐
Investiga cómo se usa el registro en imágenes médicas (ej: alinear MRI y CT scans).

In [ ]:
# ESPACIO PARA TUS EXPERIMENTOS
# Usa este espacio para probar las funciones y completar los ejercicios

# Ejemplo: Añadir ruido y evaluar robustez
def anadir_ruido_gaussiano(imagen, sigma=10):
    """Añade ruido gaussiano a una imagen."""
    ruido = np.random.normal(0, sigma, imagen.shape)
    imagen_ruidosa = np.clip(imagen + ruido, 0, 255).astype(np.uint8)
    return imagen_ruidosa

# Tu código aquí...

## 7. Recursos Adicionales

### Documentación:
- [OpenCV: Image Registration](https://docs.opencv.org/master/d5/d6f/tutorial_feature_flann_matcher.html)
- [Scikit-image: Registration](https://scikit-image.org/docs/stable/auto_examples/registration/)

### Papers Clásicos:
- Lowe, D.G. (2004). "Distinctive Image Features from Scale-Invariant Keypoints" (SIFT)
- Rublee et al. (2011). "ORB: An efficient alternative to SIFT or SURF"
- Fischler & Bolles (1981). "Random Sample Consensus" (RANSAC)

### Libros:
- Szeliski, R. "Computer Vision: Algorithms and Applications" - Capítulo 9
- Forsyth & Ponce "Computer Vision: A Modern Approach"

---

**¡Fin del Tutorial!** 🎉

Has aprendido los conceptos fundamentales del registro de imágenes y cómo implementarlos en Python. ¡Ahora es tu turno de experimentar!